# 01.7 — Dataclasses and typed configs

Python's answer to MATLAB's `inputParser` / `CheckVararginPairs` boilerplate is a combination of three modern features:

* **Type hints** — describe argument and return types so tools (pyright) catch bugs without running the code.
* **`@dataclass`** — auto-generate constructors and equality for value-like classes.
* **OmegaConf / Hydra** — schema-driven configs loaded from YAML, accessed like Python objects.

Together these give you the "typed config" experience that MATLAB users sometimes miss. This notebook covers all three.

**Prerequisite:** [01.4 classes and OOP](01.4_classes_and_oop.ipynb).

## Section 1 — What MATLAB does

A MATLAB function that accepts a long config typically opens with `inputParser` or — in this codebase — `CheckVararginPairs`:

```matlab
function out = build_model(varargin)
    name = CheckVararginPairs('name', 'GRU', varargin{:});
    hidden = CheckVararginPairs('hidden', 250, varargin{:});
    dropout = CheckVararginPairs('dropout', 0.5, varargin{:});
    is_variational = CheckVararginPairs('is_variational', true, varargin{:});
    out = struct('name', name, 'hidden', hidden, ...
                 'dropout', dropout, 'is_variational', is_variational);
end
```

Every field needs a line. Every caller has to remember which fields exist. There's no static checking — a typo (`is_varational`) silently uses the default.

## Section 2 — The Python concepts you need

### 2.1 — Type hints (PEP 484)

Type hints are annotations on parameters and return values. They don't enforce anything at runtime — Python doesn't check them. But **pyright** (the project's static checker) reads them and catches bugs at edit time.

In [ ]:
# A function with type hints
def add(x: int, y: int) -> int:
    return x + y

print(add(2, 3))       # OK — pyright sees both args are ints
# print(add("hi", 3))  # pyright would flag: "hi" is str, expected int

**The annotations are documentation that the language understands.** Without them, every reader has to guess what `hidden_sizes` should be — a list of ints? a single int? a string? With them, `hidden_sizes: list[int]` answers the question instantly.

In [ ]:
# More elaborate hints
from collections.abc import Iterable
from typing import Any

def build_classifier(
    name: str,
    hidden_sizes: list[int],
    dropout: float = 0.5,
    extras: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Build a classifier config dict."""
    cfg: dict[str, Any] = {"name": name, "hidden_sizes": hidden_sizes, "dropout": dropout}
    if extras:
        cfg.update(extras)
    return cfg

print(build_classifier("Deep LSTM - Dropout 0.5", [250, 100, 50]))

**Common annotations you'll see in this codebase:**

| Annotation | Means |
|---|---|
| `int`, `float`, `str`, `bool` | exactly what they look like |
| `list[int]` | a list of ints |
| `dict[str, Any]` | a dict from string keys to anything |
| `tuple[int, int]` | a 2-element tuple of ints |
| `int \| None` | int or None (Python 3.10+ syntax for `Optional[int]`) |
| `Path` (from pathlib) | a filesystem path |
| `torch.Tensor` | a PyTorch tensor |

The `from __future__ import annotations` line at the top of every module enables forward references so you can write `tuple[int, int]` instead of `Tuple[int, int]` even on older Python versions.

### 2.2 — `@dataclass` — typed config classes

A dataclass is a class whose fields are described by type-annotated attributes. The `__init__` is auto-generated.

In [ ]:
from dataclasses import dataclass

@dataclass
class ClassifierConfig:
    name: str
    hidden_sizes: list[int]
    dropout: float = 0.5
    is_variational: bool = True

cfg = ClassifierConfig(
    name="Deep LSTM - Dropout 0.5",
    hidden_sizes=[250, 100, 50],
)
print(cfg)               # auto __repr__
print(cfg.name)
print(cfg.dropout)        # default value picked up

Compare the MATLAB pattern (every field re-declared in every caller) vs the dataclass (declared once, used as-is). The dataclass version:

* Catches typos at edit time (`cfg.is_varational` → pyright flags it).
* Gives you `__init__`, `__repr__`, `__eq__` for free.
* Reads like a schema — anyone scanning the class definition knows what fields exist.

The codebase uses dataclasses everywhere a struct-like value is needed. Examples: `SweepEntry`, `MatlabRunDirs`, `RunBannerData`, `GitState`, `GpuInfo`.

### 2.3 — `frozen=True` + `slots=True` — the codebase default

For config-like classes where you compare instances by content and never mutate them after construction, add `frozen=True`:

In [ ]:
@dataclass(frozen=True, slots=True)
class SweepEntry:
    sweep_index: int
    matlab_choice: int
    description: str

e1 = SweepEntry(sweep_index=1, matlab_choice=1, description="Feedforward")
e2 = SweepEntry(sweep_index=1, matlab_choice=1, description="Feedforward")
print(e1 == e2)        # True — equality compares by field content
print(hash(e1))        # frozen dataclasses are hashable (can be in a set / dict key)

In [ ]:
# frozen=True rejects assignment after construction
try:
    e1.description = "Changed"    # raises FrozenInstanceError
except Exception as ex:
    print(f"  {type(ex).__name__}: {ex}")

**Why pick which flags:**

| Flag | Effect | When to use |
|---|---|---|
| `frozen=True` | reject post-init assignment | value-like classes, want hash + equality |
| `slots=True` | reject unknown attributes, faster + smaller | always (catches typos at assignment time) |
| `kw_only=True` | force callers to use keyword args | long arg lists where positional order is confusing |
| `eq=False` | don't auto-generate `__eq__` | rare; only if you want identity comparison |

The codebase default for new dataclasses is **`@dataclass(frozen=True, slots=True)`**.

### 2.4 — Default factories for mutable defaults

You CAN'T write `field: list = []` because that hits the mutable-default-argument trap (see [00.3](../00_orientation/00.3_the_matlab_to_python_mental_model.ipynb)). The fix: `field(default_factory=list)` — a callable that creates a fresh default per instance.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Run:
    name: str
    notes: list[str] = field(default_factory=list)   # safe: fresh list each time
    overrides: dict[str, int] = field(default_factory=dict)

r1 = Run(name="A")
r2 = Run(name="B")
r1.notes.append("alpha")
print(r1.notes)   # ['alpha']
print(r2.notes)   # [] — NOT shared

### 2.5 — OmegaConf / Hydra — runtime typed configs

Dataclasses cover Python-side typed values. For configs loaded from YAML, this codebase uses **OmegaConf** (and Hydra on top of it). The result feels like accessing attributes on an object, but the schema lives in a YAML file.

In [ ]:
from omegaconf import OmegaConf

# Build a config from a dict (or load from a YAML file)
cfg = OmegaConf.create({
    "model_name": "GRU",
    "hidden_sizes": [1000, 500, 250],
    "dropout": 0.5,
    "nested": {"alpha": 1.0, "beta": 2.0},
})

print(cfg.model_name)              # attribute-style access
print(cfg.hidden_sizes[0])          # indexing into nested lists
print(cfg.nested.alpha)             # nested attribute access

In [ ]:
# Get with a default (KeyError-safe)
print(cfg.get("dropout", 0.0))        # 0.5 — the actual value
print(cfg.get("missing", 99))         # 99 — the default

**Hydra's missing-mandatory `???` sentinel** appears in our `real_data_base.yaml` for `data_dir: ???` and `target_dir: ???`. Accessing a `???` field raises `MissingMandatoryValue`. This is how the config layer enforces "you must override these at sbatch time."

The CLI's `_real_data_path_active` helper wraps the access in a try/except so a missing `data_dir` falls through to the synthetic path naturally.

## Section 3 — The `neural_data_decoding` implementation

Look at the `SweepEntry` dataclass — every concept in this notebook in one definition:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/sweeps/dispatcher.py").read_text().splitlines()
start = next((i for i, line in enumerate(src) if line.startswith("@dataclass(frozen=True")), -1)
for i in range(start, min(start + 35, len(src))):
    print(f"{i + 1:3} | {src[i]}")

Things to spot:

* `@dataclass(frozen=True, slots=True)` — the codebase default.
* Type hints on every field: `sweep_index: int`, `description: str`, `overrides: dict[str, Any]`, `notes: tuple[str, ...]`.
* `default_factory=dict` for the mutable default.
* Default `()` for `notes` — a tuple is immutable so this is safe as a bare default.
* The triple-quoted docstring describes each field — used by tools that auto-generate documentation.

## Section 4 — Hands-on exercises

### Exercise 4.1 — port MATLAB inputParser to a dataclass

Port:

```matlab
function out = make_run(varargin)
    sweep_idx = CheckVararginPairs('sweep_idx', 1, varargin{:});
    fold = CheckVararginPairs('fold', 1, varargin{:});
    session = CheckVararginPairs('session', '', varargin{:});
    out = struct('sweep_idx', sweep_idx, 'fold', fold, 'session', session);
end
```

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Run:
    sweep_idx: int = 1
    fold: int = 1
    session: str = ""

r = Run(sweep_idx=91)
print(r)
print(r.fold)

### Exercise 4.2 — add a validation method

Extend the `Run` class with a `validate()` method that raises `ValueError` if `fold < 1`. Try calling it both ways.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
@dataclass(frozen=True, slots=True)
class ValidatedRun:
    sweep_idx: int = 1
    fold: int = 1
    session: str = ""

    def validate(self) -> None:
        if self.fold < 1:
            raise ValueError(f"fold must be >= 1 (MATLAB 1-indexed); got {self.fold}")

ValidatedRun(fold=2).validate()  # OK
try:
    ValidatedRun(fold=0).validate()
except ValueError as e:
    print(f"caught: {e}")

## Section 5 — Common errors

### `TypeError: __init__() got an unexpected keyword argument 'foo'`

You passed `foo=...` to a dataclass that doesn't declare `foo`. Either fix the call or add `foo` to the dataclass.

### `dataclasses.FrozenInstanceError: cannot assign to field 'x'`

You used `frozen=True` but tried to mutate. Either remove `frozen=True` if mutation is needed, OR use `dataclasses.replace(obj, x=new_value)` to build a new instance with the updated field.

### Mutable default in dataclass

```python
@dataclass
class Bad:
    items: list = []     # ValueError at class definition time!
```

Use `field(default_factory=list)`.

### `MissingMandatoryValue: Missing mandatory value: data_dir`

OmegaConf's `???` sentinel was not overridden. Pass `--override data_dir=...` at the CLI, or set the field in your YAML override.

### Type hint isn't enforced at runtime

```python
def add(x: int, y: int) -> int:
    return x + y

add("hi", 3)    # runs and raises TypeError on "hi" + 3 — but the hint didn't catch it
```

Type hints are checked by **pyright** at edit time, not by Python at runtime. Run `python -m pyright src/` to catch them. The codebase's CI does this for you.

## Section 6 — Further reading

- [PEP 484 — type hints](https://peps.python.org/pep-0484/). The original proposal.
- [PEP 604 — `X | Y` syntax](https://peps.python.org/pep-0604/). The modern union syntax used in this codebase.
- [dataclasses docs](https://docs.python.org/3/library/dataclasses.html). Every option.
- [OmegaConf docs](https://omegaconf.readthedocs.io/). The config layer we use.
- [Hydra docs](https://hydra.cc/). Configuration management built on OmegaConf.
- [`pyright`](https://github.com/microsoft/pyright). The static type checker the project uses.

Next up: **[01.8 the Python standard library for MATLAB users](01.8_the_python_standard_library_for_matlab_users.ipynb)**.